In [69]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from data_loader import load_options_data
from implied_vol import implied_volatility
from greeks import delta, gamma, vega, theta

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spy_options_sample.csv"

df = load_options_data(DATA_PATH)

df["implied_vol"] = df.apply(
    lambda row: implied_volatility(
        market_price=row["mid_price"],
        S=row["underlying_price"],
        K=row["strike"],
        T=row["time_to_expiration"],
        r=row["risk_free_rate"],
        option_type=row["option_type"],
        q=0.0,
    ),
    axis=1,
)

In [70]:
df["delta"] = delta(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"], df["option_type"])
df["gamma"] = gamma(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"])
df["vega"] = vega(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"])
df["theta"] = theta(df["underlying_price"], df["strike"], df["time_to_expiration"], df["risk_free_rate"], df["implied_vol"], df["option_type"])

df[["strike", "option_type", "mid_price", "implied_vol", "delta", "gamma", "vega", "theta"]]

,strike,option_type,mid_price,implied_vol,delta,gamma,vega,theta
0,580,C,25.00,0.133877,0.824566,0.010234,0.486497,-0.148373
1,590,C,17.10,0.127595,0.708916,0.014262,0.646140,-0.164838
2,600,C,10.60,0.080361,0.599212,0.019145,0.971160,-0.103990
3,610,C,6.15,0.061351,0.445022,0.021041,1.209557,-0.071218
4,620,C,3.45,0.067469,0.275876,0.016182,1.022977,-0.056308
5,580,P,4.75,0.120018,-0.219000,0.008039,0.903980,-0.040317
6,590,P,7.40,0.109096,-0.295785,0.009018,1.212979,-0.030140
7,600,P,10.30,0.099004,-0.370884,0.009730,1.482176,-0.018327
8,610,P,14.90,0.100461,-0.470504,0.010096,1.560587,-0.013608
9,620,P,20.50,0.102044,-0.567440,0.009824,1.542453,-0.005946


In [71]:
df["position"] = [10, 5, -8, -4, 3, -6, 7, 5, -3, 2]
df["contract_multiplier"] = 100

df["position_delta"] = df["position"] * df["contract_multiplier"] * df["delta"]
df["position_vega"] = df["position"] * df["contract_multiplier"] * df["vega"]
df["position_gamma"] = df["position"] * df["contract_multiplier"] * df["gamma"]
df["position_theta"] = df["position"] * df["contract_multiplier"] * df["theta"]

portfolio_exposure = df[[
    "position_delta",
    "position_vega",
    "position_gamma",
    "position_theta"
]].sum()

portfolio_exposure

position_delta    370.979301
position_vega     743.809112
position_gamma      3.778021
position_theta   -139.183820
dtype: float64

In [72]:
from quote_engine import generate_quotes

net_delta = portfolio_exposure["position_delta"]
net_vega = portfolio_exposure["position_vega"]

df["fair_value"] = df["mid_price"]

quotes = df.apply(
    lambda row: generate_quotes(
        fair_value=row["fair_value"],
        market_spread=row["bid_ask_spread"],
        implied_vol=row["implied_vol"],
        inventory=row["position"],
        net_delta=net_delta,
        net_vega=net_vega,
    ),
    axis=1,
)

quotes_df = quotes.apply(lambda x: pd.Series(x))

quotes_df = quotes_df.rename(columns={
    "bid": "quoted_bid",
    "ask": "quoted_ask"
})

df = pd.concat([df, quotes_df], axis=1)

df[[
    "strike",
    "option_type",
    "position",
    "mid_price",
    "fair_value",
    "bid",
    "ask",
    "quoted_spread",
    "quote_skew"
]]

,strike,option_type,position,mid_price,fair_value,bid,ask,quoted_spread,quote_skew
0,580,C,10,25.00,25.00,24.8,25.2,0.266939,0.174288
1,590,C,5,17.10,17.10,16.9,17.3,0.263798,0.124288
2,600,C,-8,10.60,10.60,10.4,10.8,0.240180,-0.005712
3,610,C,-4,6.15,6.15,6.0,6.3,0.205676,0.034288
4,620,C,3,3.45,3.45,3.3,3.6,0.208735,0.104288
5,580,P,-6,4.75,4.75,4.6,4.9,0.235009,0.014288
6,590,P,7,7.40,7.40,7.2,7.6,0.254548,0.144288
7,600,P,5,10.30,10.30,10.1,10.5,0.249502,0.124288
8,610,P,-3,14.90,14.90,14.7,15.1,0.250230,0.044288
9,620,P,2,20.50,20.50,20.2,20.8,0.301022,0.094288


In [73]:
from fill_sim import simulate_fill
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

fills = df.apply(
    lambda row: simulate_fill(
        quoted_bid=row["bid"].iloc[-1] if hasattr(row["bid"], "iloc") else row["bid"],
        quoted_ask=row["ask"].iloc[-1] if hasattr(row["ask"], "iloc") else row["ask"],
        fair_value=row["fair_value"],
        quoted_spread=row["quoted_spread"],
        market_spread=row["bid_ask_spread"],
        volume=row["volume"],
        rng=rng,
    ),
    axis=1,
)

fills_df = fills.apply(pd.Series)

df = pd.concat([df, fills_df], axis=1)

df[[
    "strike",
    "option_type",
    "fair_value",
    "quoted_spread",
    "fill_probability",
    "filled",
    "fill_side",
    "fill_price",
    "spread_capture"
]]

,strike,option_type,fair_value,quoted_spread,fill_probability,filled,fill_side,fill_price,spread_capture
0,580,C,25.00,0.266939,0.562988,False,None,NaN,0.00
1,590,C,17.10,0.263798,0.577091,True,buy,16.9,0.20
2,600,C,10.60,0.240180,0.647640,False,None,NaN,0.00
3,610,C,6.15,0.205676,0.560782,True,sell,6.3,0.15
4,620,C,3.45,0.208735,0.533730,False,None,NaN,0.00
5,580,P,4.75,0.235009,0.475279,False,None,NaN,0.00
6,590,P,7.40,0.254548,0.601776,False,None,NaN,0.00
7,600,P,10.30,0.249502,0.628206,True,sell,10.5,0.20
8,610,P,14.90,0.250230,0.609474,True,buy,14.7,0.20
9,620,P,20.50,0.301022,0.709645,False,None,NaN,0.00


In [74]:
from inventory_manager import InventoryManager

inventory = InventoryManager(contract_multiplier=100)

inventory.process_fill(
    symbol="SPY_20260220_600_C",
    fill_side="buy",
    fill_price=10.50,
    quantity=1,
)

inventory.process_fill(
    symbol="SPY_20260220_600_C",
    fill_side="sell",
    fill_price=10.80,
    quantity=1,
)

inventory.get_positions(), inventory.get_cash(), inventory.get_trade_log()

({'SPY_20260220_600_C': 0},
 30.0,
 [{'symbol': 'SPY_20260220_600_C',
   'side': 'buy',
   'quantity': 1,
   'fill_price': 10.5,
   'cash_change': -1050.0,
   'old_position': 0,
   'new_position': 1,
   'average_price': 10.5,
   'realized_pnl_change': 0.0,
   'cumulative_realized_pnl': 0.0,
   'cash': -1050.0},
  {'symbol': 'SPY_20260220_600_C',
   'side': 'sell',
   'quantity': 1,
   'fill_price': 10.8,
   'cash_change': 1080.0,
   'old_position': 1,
   'new_position': 0,
   'average_price': 0.0,
   'realized_pnl_change': 30.00000000000007,
   'cumulative_realized_pnl': 30.00000000000007,
   'cash': 30.0}])

In [75]:
df["symbol"] = (
    "SPY_"
    + df["expiration"].dt.strftime("%Y%m%d")
    + "_"
    + df["strike"].astype(int).astype(str)
    + "_"
    + df["option_type"]
)

df[["symbol", "strike", "option_type"]]

,symbol,strike,option_type
0,SPY_20260220_580_C,580,C
1,SPY_20260220_590_C,590,C
2,SPY_20260320_600_C,600,C
3,SPY_20260420_610_C,610,C
4,SPY_20260420_620_C,620,C
5,SPY_20260420_580_P,580,P
6,SPY_20260520_590_P,590,P
7,SPY_20260620_600_P,600,P
8,SPY_20260620_610_P,610,P
9,SPY_20260620_620_P,620,P


In [76]:
from fill_sim import simulate_fill
from inventory_manager import InventoryManager
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

inventory = InventoryManager(contract_multiplier=100)

fill_results = []

for _, row in df.iterrows():

    fill = simulate_fill(
        quoted_bid=row["quoted_bid"],
        quoted_ask=row["quoted_ask"],
        fair_value=row["fair_value"],
        quoted_spread=row["quoted_spread"],
        market_spread=row["bid_ask_spread"],
        volume=row["volume"],
        rng=rng,
    )

    if fill["filled"]:

        inventory.process_fill(
            symbol=row["symbol"],
            fill_side=fill["fill_side"],
            fill_price=fill["fill_price"],
            quantity=1,
        )

    fill_results.append(fill)

fills_df = pd.DataFrame(fill_results)

fill_columns = [
    "filled",
    "fill_side",
    "fill_price",
    "fill_probability",
    "spread_capture",
]

df = df.drop(
    columns=[column for column in fill_columns if column in df.columns]
)

df = pd.concat(
    [
        df.reset_index(drop=True),
        fills_df.reset_index(drop=True),
    ],
    axis=1,
)

df[[
    "symbol",
    "quoted_bid",
    "quoted_ask",
    "fill_probability",
    "filled",
    "fill_side",
    "fill_price",
    "spread_capture"
]]

,symbol,quoted_bid,quoted_ask,fill_probability,filled,fill_side,fill_price,spread_capture
0,SPY_20260220_580_C,24.692242,24.959181,0.562988,False,None,NaN,0.000000
1,SPY_20260220_590_C,16.843813,17.107610,0.577091,True,buy,16.843813,0.256187
2,SPY_20260320_600_C,10.485621,10.725802,0.647640,False,None,NaN,0.000000
3,SPY_20260420_610_C,6.012874,6.218549,0.560782,True,sell,6.218549,0.068549
4,SPY_20260420_620_C,3.241344,3.450079,0.533730,False,None,NaN,0.000000
5,SPY_20260420_580_P,4.618207,4.853216,0.475279,False,None,NaN,0.000000
6,SPY_20260520_590_P,7.128438,7.382986,0.601776,False,None,NaN,0.000000
7,SPY_20260620_600_P,10.050961,10.300463,0.628206,True,sell,10.300463,0.000463
8,SPY_20260620_610_P,14.730596,14.980827,0.609474,True,buy,14.730596,0.169404
9,SPY_20260620_620_P,20.255201,20.556223,0.709645,False,None,NaN,0.000000


In [77]:
inventory.get_positions()
inventory.get_cash()

-1505.5397141104236

In [78]:
pd.DataFrame(
    inventory.get_trade_log()
)

,symbol,side,quantity,fill_price,cash_change,old_position,new_position,average_price,realized_pnl_change,cumulative_realized_pnl,cash
0,SPY_20260220_590_C,buy,1,16.843813,-1684.381280,0,1,16.843813,0.0,0.0,-1684.381280
1,SPY_20260420_610_C,sell,1,6.218549,621.854948,0,-1,6.218549,0.0,0.0,-1062.526332
2,SPY_20260620_600_P,sell,1,10.300463,1030.046266,0,-1,10.300463,0.0,0.0,-32.480066
3,SPY_20260620_610_P,buy,1,14.730596,-1473.059648,0,1,14.730596,0.0,0.0,-1505.539714


In [79]:
pd.DataFrame(inventory.get_trade_log())

,symbol,side,quantity,fill_price,cash_change,old_position,new_position,average_price,realized_pnl_change,cumulative_realized_pnl,cash
0,SPY_20260220_590_C,buy,1,16.843813,-1684.381280,0,1,16.843813,0.0,0.0,-1684.381280
1,SPY_20260420_610_C,sell,1,6.218549,621.854948,0,-1,6.218549,0.0,0.0,-1062.526332
2,SPY_20260620_600_P,sell,1,10.300463,1030.046266,0,-1,10.300463,0.0,0.0,-32.480066
3,SPY_20260620_610_P,buy,1,14.730596,-1473.059648,0,1,14.730596,0.0,0.0,-1505.539714


## Mark-to-Market Portfolio PnL

This section values all open option inventory using current fair values
and combines that value with cash to calculate total portfolio equity.

In [80]:
from pnl import calculate_total_pnl

In [81]:
current_prices = (
    df.drop_duplicates(subset="symbol", keep="last")
      .set_index("symbol")["fair_value"]
      .to_dict()
)

current_prices

{'SPY_20260220_580_C': 25.0,
 'SPY_20260220_590_C': 17.1,
 'SPY_20260320_600_C': 10.600000000000001,
 'SPY_20260420_610_C': 6.15,
 'SPY_20260420_620_C': 3.45,
 'SPY_20260420_580_P': 4.75,
 'SPY_20260520_590_P': 7.4,
 'SPY_20260620_600_P': 10.3,
 'SPY_20260620_610_P': 14.899999999999999,
 'SPY_20260620_620_P': 20.5}

In [82]:
from pnl import calculate_unrealized_pnl

realized_pnl = inventory.get_realized_pnl()

unrealized_pnl = calculate_unrealized_pnl(
    positions=inventory.get_positions(),
    average_prices=inventory.get_average_prices(),
    current_prices=current_prices,
    contract_multiplier=100,
)

detailed_pnl = pd.Series({
    "realized_pnl": realized_pnl,
    "unrealized_pnl": unrealized_pnl,
    "total_pnl": realized_pnl + unrealized_pnl,
})

detailed_pnl

realized_pnl       0.000000
unrealized_pnl    49.460286
total_pnl         49.460286
dtype: float64

In [83]:
pnl_summary = calculate_total_pnl(
    cash=inventory.get_cash(),
    positions=inventory.get_positions(),
    current_prices=current_prices,
    contract_multiplier=100,
    initial_equity=0.0,
)

pnl_summary

{'cash': -1505.5397141104236,
 'market_value': 1555.0,
 'total_equity': 49.460285889576426,
 'total_pnl': 49.460285889576426}

In [84]:
import pandas as pd

pd.Series(pnl_summary, name="portfolio")

cash           -1505.539714
market_value    1555.000000
total_equity      49.460286
total_pnl         49.460286
Name: portfolio, dtype: float64

In [85]:
from inventory_manager import InventoryManager
import pandas as pd

test_inventory = InventoryManager(contract_multiplier=100)

test_inventory.process_fill(
    symbol="SPY_TEST_600_C",
    fill_side="buy",
    fill_price=10.00,
    quantity=2,
)

test_inventory.process_fill(
    symbol="SPY_TEST_600_C",
    fill_side="sell",
    fill_price=12.00,
    quantity=1,
)

pd.DataFrame(test_inventory.get_trade_log())

,symbol,side,quantity,fill_price,cash_change,old_position,new_position,average_price,realized_pnl_change,cumulative_realized_pnl,cash
0,SPY_TEST_600_C,buy,2,10.0,-2000.0,0,2,10.0,0.0,0.0,-2000.0
1,SPY_TEST_600_C,sell,1,12.0,1200.0,2,1,10.0,200.0,200.0,-800.0


In [86]:
pnl_summary["total_pnl"]

accounting_difference = (
    detailed_pnl["total_pnl"]
    - pnl_summary["total_pnl"]
)

accounting_difference

4.263256414560601e-14